<a href="https://colab.research.google.com/github/jcmachicao/cur_MMAI__multimodal/blob/main/cur_IAGenMM___vectorizacionMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

In [ ]:
ruta_archivo = "drive/My Drive/00 data/bike img/"
os.listdir(ruta_archivo)

In [ ]:
# CARGAR JSON
with open(ruta_archivo + "textos_multimodales.json", "r", encoding="utf-8") as f:
    data = json.load(f)

items = data["items"]
items[0]

* SENTENCE TRANSFORMERS
* TOPIC + KEYWORDS
* COLOR = TOPIC
* MARCADOR = TYPE

In [ ]:
# TEXTOS EMBEDDINGS PCA CLUSTERING COLORES POR TOPIC
texts = [item["text"] for item in items]

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts)
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)
kmeans = KMeans(n_clusters=6, random_state=42)
clusters = kmeans.fit_predict(embeddings)

topic_colors = {
    "accidente_interseccion": "red",
    "infraestructura_ciclovia": "blue",
    "riesgo_vehicular": "orange",
    "movilidad_recreativa": "green",
    "control_municipal": "purple",
    "convivencia_vial": "brown",
    "seguridad_preventiva": "pink",
    "mantenimiento_urbano": "gray",
    "seguridad_nocturna": "cyan"
}

type_markers = {
    "descripcion_visual": "o",
    "reglamento": "s",
    "reporte": "^"
}

# PLOT
plt.figure(figsize=(18, 14))

for i, item in enumerate(items):

    topic = item["topic"]
    doc_type = item["type"]

    color = topic_colors.get(topic, "black")
    marker = type_markers.get(doc_type, "x")

    x = reduced[i][0]
    y = reduced[i][1]

    plt.scatter(
        x,
        y,
        c=color,
        marker=marker,
        s=140
    )

    # LABELS
    # topic en primera línea
    # keywords en segunda línea

    keywords = ", ".join(item["keywords"][:3])

    label = f"{topic}\n{keywords}"

    plt.text(
        x + 0.01,
        y + 0.01,
        label,
        fontsize=8
    )

plt.title("Sentence Transformers - Espacio Semántico Multimodal")

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.patches as mpatches

# PLOT (re-creating the plot to add legends)
plt.figure(figsize=(18, 14))

# Dictionaries to store handles for legends
topic_legend_handles = {}
type_legend_handles = {}

for i, item in enumerate(items):

    topic = item["topic"]
    doc_type = item["type"]

    color = topic_colors.get(topic, "black")
    marker = type_markers.get(doc_type, "x")

    x = reduced[i][0]
    y = reduced[i][1]

    # Plot the point
    plt.scatter(
        x,
        y,
        c=color,
        marker=marker,
        s=140,
        label=f"Topic: {topic}" if topic not in topic_legend_handles else "_nolegend_"
    )

    # Store dummy handles for topic colors for the legend
    if topic not in topic_legend_handles:
        topic_legend_handles[topic] = mpatches.Patch(color=color, label=topic)

    # Store dummy handles for type markers for the legend
    if doc_type not in type_legend_handles:
        # For markers, we plot a single invisible point to get the legend handle
        type_legend_handles[doc_type] = plt.scatter([], [], marker=marker, color='gray', label=doc_type)

    # LABELS
    # topic en primera línea
    # keywords en segunda línea

    keywords = ", ".join(item["keywords"][:3])

    label_text = f"{topic}\n{keywords}"

    plt.text(
        x + 0.01,
        y + 0.01,
        label_text,
        fontsize=8
    )

plt.title("Sentence Transformers - Espacio Semántico Multimodal")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(True)

# Create legend for topics (colors)
topic_legend = plt.legend(
    handles=list(topic_legend_handles.values()),
    title="Topics",
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    borderaxespad=0.
)

# Add the topic legend to the current axes
plt.gca().add_artist(topic_legend)

# Create legend for types (markers)
type_legend = plt.legend(
    handles=[handle for handle in type_legend_handles.values()],
    labels=[label for label in type_legend_handles.keys()],
    title="Document Types",
    bbox_to_anchor=(1.02, 0.6),
    loc='upper left',
    borderaxespad=0.
)

plt.tight_layout()
plt.show()